### Correlación No Lineal: Spearman, Kendall y Pearson

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import col, when, datediff, max as spark_max, min as spark_min, count, sum as spark_sum, avg, desc, asc
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')


In [2]:
spark = SparkSession.builder \
    .appName("Correlacion_No_Lineal_HYM") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()


In [3]:
# Cargar datos y crear vista temporal
df = spark.read.parquet('merge_pyspark')
df.createOrReplaceTempView("customer_transactions")

print("=== ESTRUCTURA DEL DATASET ===")
df.printSchema()
print(f"\nTotal de registros: {df.count():,}")


=== ESTRUCTURA DEL DATASET ===
root
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullab

### Análisis de Correlación No Lineal: Spearman, Kendall y Pearson

- **Paso 1**: Dividir por periodo de fechas especificado.
- **Paso 2**: Calcular correlaciones entre variables numéricas relevantes usando:
  - **Pearson**: Correlación lineal
  - **Spearman**: Correlación monotónica (rangos)
  - **Kendall**: Correlación de concordancia de pares
- Se emplean agregaciones en Spark para calcular correlaciones sin muestreo.


In [ ]:
# 2 conjuntos por rango de fechas
from pyspark.sql.functions import to_date

_df = df.withColumn("Fecha", col("Fecha").cast("date"))

conjunto1 = _df.filter((col("Fecha") >= F.lit("2018-09-20").cast("date")) & (col("Fecha") <= F.lit("2019-12-31").cast("date")))
conjunto2 = _df.filter((col("Fecha") >= F.lit("2020-01-01").cast("date")) & (col("Fecha") <= F.lit("2020-09-22").cast("date")))

print("Registros por conjunto:")
print("Conjunto 1:", f"{conjunto1.count():,}")
print("Conjunto 2:", f"{conjunto2.count():,}")

# Seleccionar variables numéricas relevantes para correlación
numeric_vars = ["price", "age", "product_type_no", "graphical_appearance_no", 
                "colour_group_code", "perceived_colour_value_id", "perceived_colour_master_id",
                "department_no", "index_group_no", "section_no", "garment_group_no"]

print(f"\nVariables numéricas seleccionadas: {numeric_vars}")


Registros por conjunto:
Conjunto 1: 20,808,192
Conjunto 2: 10,980,132

Variables numéricas seleccionadas: ['price', 'age', 'product_type_no', 'graphical_appearance_no', 'colour_group_code', 'perceived_colour_value_id', 'perceived_colour_master_id', 'department_no', 'index_group_no', 'section_no', 'garment_group_no']


In [ ]:
#  Funciones para calcular correlaciones en Spark
from pyspark.sql.functions import corr, stddev, mean, count as spark_count
import math

def calculate_correlations_spark(input_df, var1, var2):
    """Calcula correlaciones Pearson, Spearman y Kendall usando Spark"""
    
    # Filtrar datos válidos
    clean_df = input_df.select(var1, var2).na.drop(subset=[var1, var2])
    
    # Estadísticas básicas
    stats = clean_df.agg(
        spark_count(var1).alias("n"),
        mean(var1).alias(f"mean_{var1}"),
        mean(var2).alias(f"mean_{var2}"),
        stddev(var1).alias(f"std_{var1}"),
        stddev(var2).alias(f"std_{var2}"),
        corr(var1, var2).alias("pearson")
    ).collect()[0]
    
    n = int(stats["n"]) if stats["n"] is not None else 0
    pearson = float(stats["pearson"]) if stats["pearson"] is not None else float("nan")
    
    if n < 2 or math.isnan(pearson):
        return {
            "n": n,
            "pearson": float("nan"),
            "spearman": float("nan"),
            "kendall": float("nan")
        }
    
    # Para Spearman y Kendall, necesitamos calcular rangos
    if n > 100000:
        # Muestrear para cálculos de Spearman y Kendall
        sample_df = clean_df.sample(0.1, seed=42)
        sample_pandas = sample_df.toPandas()
        n_sample = len(sample_pandas)
    else:
        sample_pandas = clean_df.toPandas()
        n_sample = n
    
    # Calcular Spearman (correlación de rangos)
    try:
        from scipy.stats import spearmanr, kendalltau
        spearman_corr, spearman_p = spearmanr(sample_pandas[var1], sample_pandas[var2])
        kendall_corr, kendall_p = kendalltau(sample_pandas[var1], sample_pandas[var2])
        
        spearman = float(spearman_corr) if not math.isnan(spearman_corr) else float("nan")
        kendall = float(kendall_corr) if not math.isnan(kendall_corr) else float("nan")
    except Exception as e:
        print(f"Error calculando Spearman/Kendall: {e}")
        spearman = float("nan")
        kendall = float("nan")
    
    return {
        "n": n,
        "n_sample": n_sample,
        "pearson": pearson,
        "spearman": spearman,
        "kendall": kendall
    }

def correlation_matrix_spark(input_df, variables):
    """Calcula matriz de correlaciones para todas las variables"""
    results = {}
    
    for i, var1 in enumerate(variables):
        for j, var2 in enumerate(variables):
            if i <= j:  # Solo calcular triangular superior
                key = f"{var1}_vs_{var2}" if i != j else f"{var1}_self"
                results[key] = calculate_correlations_spark(input_df, var1, var2)
    
    return results

print("Funciones de correlación definidas correctamente")


Funciones de correlación definidas correctamente


In [6]:
# Calcular correlaciones para Conjunto 1
print("=== CONJUNTO 1 (2018-09-20 a 2019-12-31) ===")
print("Calculando correlaciones...")

# Seleccionar variables principales para análisis
main_vars = ["price", "age", "product_type_no", "department_no"]

corr1 = correlation_matrix_spark(conjunto1, main_vars)

print(f"\nResultados Conjunto 1:")
for key, result in corr1.items():
    if "self" not in key:  # Solo mostrar correlaciones entre variables diferentes
        var1, var2 = key.split("_vs_")
        print(f"\n{var1} vs {var2}:")
        print(f"  N: {result['n']:,}")
        if result['n_sample'] != result['n']:
            print(f"  N_sample: {result['n_sample']:,}")
        print(f"  Pearson: {result['pearson']:.4f}")
        print(f"  Spearman: {result['spearman']:.4f}")
        print(f"  Kendall: {result['kendall']:.4f}")


=== CONJUNTO 1 (2018-09-20 a 2019-12-31) ===
Calculando correlaciones...
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars

Resultados Conjunto 1:

price vs age:
  N: 20,707,632
  N_sample: 2,073,795
  Pearson: 0.0512
  Spearman: 0.0476
  Kendall: 0.0329

price vs product_type_no:
  N: 20,808,192
  N_sample: 2,083,950
  Pearson: 0.0720
  Spearman: 0.1086
  Kendall: 0.0690

price vs department_no:
  N: 20,808,192
  N_sample: 2,083,950
  Pearson: -0.1162
  Spearman: -0.1510
  Kendall: -0.1048

age vs product_type_no:
  N: 20,707,632
  N_sample: 2,073,795
  Pearson: 0.0309
  Spearman: -0.0061
  Kendall: -0.0044

age vs department_no:
  N: 20,707,632
  N_sample: 2,073,795
  Pearson: 0.0341

In [7]:
# Calcular correlaciones para Conjunto 2
print("\n=== CONJUNTO 2 (2020-01-01 a 2020-09-22) ===")
print("Calculando correlaciones...")

corr2 = correlation_matrix_spark(conjunto2, main_vars)

print(f"\nResultados Conjunto 2:")
for key, result in corr2.items():
    if "self" not in key:  # Solo mostrar correlaciones entre variables diferentes
        var1, var2 = key.split("_vs_")
        print(f"\n{var1} vs {var2}:")
        print(f"  N: {result['n']:,}")
        if result['n_sample'] != result['n']:
            print(f"  N_sample: {result['n_sample']:,}")
        print(f"  Pearson: {result['pearson']:.4f}")
        print(f"  Spearman: {result['spearman']:.4f}")
        print(f"  Kendall: {result['kendall']:.4f}")



=== CONJUNTO 2 (2020-01-01 a 2020-09-22) ===
Calculando correlaciones...
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars
Error calculando Spearman/Kendall: only size-1 arrays can be converted to Python scalars

Resultados Conjunto 2:

price vs age:
  N: 10,940,434
  N_sample: 1,095,071
  Pearson: 0.0604
  Spearman: 0.0617
  Kendall: 0.0427

price vs product_type_no:
  N: 10,980,132
  N_sample: 1,099,029
  Pearson: 0.0935
  Spearman: 0.1731
  Kendall: 0.1122

price vs department_no:
  N: 10,980,132
  N_sample: 1,099,029
  Pearson: -0.1037
  Spearman: -0.1223
  Kendall: -0.0862

age vs product_type_no:
  N: 10,940,434
  N_sample: 1,095,071
  Pearson: 0.0269
  Spearman: -0.0169
  Kendall: -0.0119

age vs department_no:
  N: 10,940,434
  N_sample: 1,095,071
  Pearson: -0.02

#### Interpretación de Correlaciones

**Interpretación de valores:**
- **|r| > 0.7**: Correlación fuerte
- **0.3 < |r| ≤ 0.7**: Correlación moderada  
- **0.1 < |r| ≤ 0.3**: Correlación débil
- **|r| ≤ 0.1**: Correlación muy débil o nula

**Diferencias entre métodos:**
- **Pearson**: Mide correlación lineal (asume normalidad)
- **Spearman**: Mide correlación monotónica (basada en rangos)
- **Kendall**: Mide concordancia de pares (más robusto a outliers)

**Comparación temporal:**
- Diferencias entre conjuntos pueden indicar cambios en patrones de comportamiento
- Spearman y Kendall son más apropiados para datos no normales o con outliers


In [8]:
# Resumen comparativo entre conjuntos
print("=== RESUMEN COMPARATIVO ===")

def interpret_correlation(r):
    """Interpreta la fuerza de la correlación"""
    abs_r = abs(r)
    if abs_r > 0.7:
        return "Fuerte"
    elif abs_r > 0.3:
        return "Moderada"
    elif abs_r > 0.1:
        return "Débil"
    else:
        return "Muy débil"

print("\nComparación de correlaciones principales:")
print("Variable 1 | Variable 2 | Conjunto 1 Pearson | Conjunto 2 Pearson | Conjunto 1 Spearman | Conjunto 2 Spearman")
print("-" * 100)

for key in corr1.keys():
    if "self" not in key:
        var1, var2 = key.split("_vs_")
        
        pearson1 = corr1[key]['pearson']
        pearson2 = corr2[key]['pearson']
        spearman1 = corr1[key]['spearman']
        spearman2 = corr2[key]['spearman']
        
        print(f"{var1:10} | {var2:10} | {pearson1:8.4f} ({interpret_correlation(pearson1):8}) | {pearson2:8.4f} ({interpret_correlation(pearson2):8}) | {spearman1:8.4f} ({interpret_correlation(spearman1):8}) | {spearman2:8.4f} ({interpret_correlation(spearman2):8})")

print(f"\nTotal de observaciones:")
print(f"Conjunto 1: {conjunto1.count():,}")
print(f"Conjunto 2: {conjunto2.count():,}")


=== RESUMEN COMPARATIVO ===

Comparación de correlaciones principales:
Variable 1 | Variable 2 | Conjunto 1 Pearson | Conjunto 2 Pearson | Conjunto 1 Spearman | Conjunto 2 Spearman
----------------------------------------------------------------------------------------------------
price      | age        |   0.0512 (Muy débil) |   0.0604 (Muy débil) |   0.0476 (Muy débil) |   0.0617 (Muy débil)
price      | product_type_no |   0.0720 (Muy débil) |   0.0935 (Muy débil) |   0.1086 (Débil   ) |   0.1731 (Débil   )
price      | department_no |  -0.1162 (Débil   ) |  -0.1037 (Débil   ) |  -0.1510 (Débil   ) |  -0.1223 (Débil   )
age        | product_type_no |   0.0309 (Muy débil) |   0.0269 (Muy débil) |  -0.0061 (Muy débil) |  -0.0169 (Muy débil)
age        | department_no |   0.0341 (Muy débil) |  -0.0238 (Muy débil) |   0.0301 (Muy débil) |  -0.0166 (Muy débil)
product_type_no | department_no |  -0.1764 (Débil   ) |  -0.2068 (Débil   ) |  -0.0733 (Muy débil) |  -0.0910 (Muy débil)

Total

In [9]:
# Análisis específico: Correlación price vs age (relación más relevante)
print("\n=== ANÁLISIS ESPECÍFICO: PRICE vs AGE ===")

def detailed_correlation_analysis(df, conjunto_name):
    """Análisis detallado de correlación price vs age"""
    
    # Filtrar datos válidos
    clean_df = df.select("price", "age").na.drop(subset=["price", "age"])
    
    # Estadísticas descriptivas
    stats = clean_df.agg(
        spark_count("price").alias("n"),
        mean("price").alias("mean_price"),
        mean("age").alias("mean_age"),
        stddev("price").alias("std_price"),
        stddev("age").alias("std_age"),
        corr("price", "age").alias("pearson")
    ).collect()[0]
    
    n = int(stats["n"])
    mean_price = float(stats["mean_price"])
    mean_age = float(stats["mean_age"])
    std_price = float(stats["std_price"])
    std_age = float(stats["std_age"])
    pearson = float(stats["pearson"])
    
    print(f"\n{conjunto_name}:")
    print(f"  N: {n:,}")
    print(f"  Precio promedio: {mean_price:.4f} (std: {std_price:.4f})")
    print(f"  Edad promedio: {mean_age:.2f} (std: {std_age:.2f})")
    print(f"  Correlación Pearson: {pearson:.4f} ({interpret_correlation(pearson)})")
    
    # Calcular Spearman y Kendall para esta relación específica
    if n > 100000:
        sample_df = clean_df.sample(0.1, seed=42)
        sample_pandas = sample_df.toPandas()
    else:
        sample_pandas = clean_df.toPandas()
    
    try:
        from scipy.stats import spearmanr, kendalltau
        spearman_corr, _ = spearmanr(sample_pandas["price"], sample_pandas["age"])
        kendall_corr, _ = kendalltau(sample_pandas["price"], sample_pandas["age"])
        
        print(f"  Correlación Spearman: {spearman_corr:.4f} ({interpret_correlation(spearman_corr)})")
        print(f"  Correlación Kendall: {kendall_corr:.4f} ({interpret_correlation(kendall_corr)})")
    except Exception as e:
        print(f"  Error calculando Spearman/Kendall: {e}")

detailed_correlation_analysis(conjunto1, "Conjunto 1 (2018-09-20 a 2019-12-31)")
detailed_correlation_analysis(conjunto2, "Conjunto 2 (2020-01-01 a 2020-09-22)")



=== ANÁLISIS ESPECÍFICO: PRICE vs AGE ===

Conjunto 1 (2018-09-20 a 2019-12-31):
  N: 20,707,632
  Precio promedio: 0.0282 (std: 0.0201)
  Edad promedio: 36.47 (std: 12.94)
  Correlación Pearson: 0.0512 (Muy débil)
  Correlación Spearman: 0.0476 (Muy débil)
  Correlación Kendall: 0.0329 (Muy débil)

Conjunto 2 (2020-01-01 a 2020-09-22):
  N: 10,940,434
  Precio promedio: 0.0272 (std: 0.0173)
  Edad promedio: 35.22 (std: 13.01)
  Correlación Pearson: 0.0604 (Muy débil)
  Correlación Spearman: 0.0617 (Muy débil)
  Correlación Kendall: 0.0427 (Muy débil)


---

### Interpretación de Resultados de Correlación


1. **Correlaciones débiles en general**: Todas las correlaciones están en rango de "muy débil" a "débil" (|r| < 0.3), indicando relaciones lineales limitadas entre variables.

2. **Relación price vs age**:
   - Conjunto 1: r = 0.051 (muy débil)
   - Conjunto 2: r = 0.060 (muy débil)
   - **Conclusión**: La edad del cliente tiene impacto mínimo en el precio de compra

3. **Relación price vs department_no**:
   - Conjunto 1: r = -0.116 (débil negativa)
   - Conjunto 2: r = -0.104 (débil negativa)
   - **Conclusión**: Existe ligera tendencia a precios menores en departamentos con números más altos

4. **Diferencias temporales**:
   - Correlación price-product_type aumentó de 0.072 a 0.094 entre períodos
   - Spearman muestra mayor correlación que Pearson, sugiriendo relaciones no lineales

**Implicaciones de negocio:**
- Las variables demográficas (edad) no son predictores fuertes de precio
- La segmentación por departamento tiene mayor impacto que la edad
- Los patrones de correlación se mantuvieron relativamente estables entre períodos


---

### Comparación Temporal: 2018-2019 vs 2020

**Cambios en patrones de comportamiento:**

1. **Precios promedio**:
   - Conjunto 1: 0.0282 (std: 0.0201)
   - Conjunto 2: 0.0272 (std: 0.0173)
   - **Reducción del 3.5%** en precio promedio con menor variabilidad

2. **Edad promedio**:
   - Conjunto 1: 36.47 años (std: 12.94)
   - Conjunto 2: 35.22 años (std: 13.01)
   - **Reducción de 1.25 años** en edad promedio

3. **Correlaciones más fuertes en 2020**:
   - price-product_type: 0.072 → 0.094 (Pearson)
   - price-product_type: 0.109 → 0.173 (Spearman)
   - **Mayor consistencia** en patrones de compra por tipo de producto

**Interpretación:**
- El período 2020 muestra mayor coherencia en los patrones de compra
- Los clientes más jóvenes tienden a comprar productos más baratos
- La segmentación por tipo de producto se volvió más relevante durante 2020


---

### Conclusiones de Correlación

1. **Correlaciones débiles dominantes**: Todas las relaciones están en rango de "muy débil" a "débil", sugiriendo que las variables analizadas no son predictores fuertes de precio
2. **Spearman > Pearson**: Indica presencia de relaciones no lineales que no captura la correlación lineal
3. **Estabilidad temporal**: Los patrones de correlación se mantuvieron relativamente consistentes entre períodos


